# 05 â€” Hyperparameter Tuning

Optuna search on the winning model from 03. Bayesian search beats grid search at the same compute. 15 trials is enough to see if there's meaningful headroom; 50+ is for production.

## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc',
]
CATEGORICAL = ['term', 'grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state']

def parse_term(s):
    if isinstance(s, str):
        return int(s.strip().split()[0])
    return np.nan

def parse_emp_length(s):
    if not isinstance(s, str):
        return np.nan
    s = s.strip()
    if '<' in s:
        return 0
    if '+' in s:
        return 10
    parts = s.split()
    return int(parts[0]) if parts and parts[0].isdigit() else np.nan

def parse_pct(s):
    if isinstance(s, str):
        s = s.replace('%', '').strip()
        return float(s) if s else np.nan
    return s

# Load and clean
df = pd.read_csv('../data/loan.csv', low_memory=False)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['target'] = (df['loan_status'] == 'Charged Off').astype(int)
df, _ = train_test_split(df, train_size=0.10, stratify=df['target'], random_state=42)

df['term'] = df['term'].map(parse_term)
df['int_rate'] = df['int_rate'].map(parse_pct)
df['revol_util'] = df['revol_util'].map(parse_pct)
df['emp_length'] = df['emp_length'].map(parse_emp_length)

# Data-quality fixes (per notebook 01 findings)
df.loc[df['dti'] > 50, 'dti'] = np.nan
df.loc[df['revol_util'] > 100, 'revol_util'] = np.nan

X = df[FEATURES].copy()
for col in CATEGORICAL:
    X[col] = X[col].astype('category')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}  Default rate: {y_train.mean():.1%}')


Train: 107,624  Test: 26,907  Default rate: 20.0%


## Optuna search over LightGBM hyperparameters

In [2]:
import optuna
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Use a holdout from the train set for the tuning objective
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': 5,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, proba)

print('Running Optuna (15 trials)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15, show_progress_bar=False)
print(f'\nBest validation AUC: {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

# Retrain on full training set with best params and evaluate on test set
final_model = lgb.LGBMClassifier(**{**study.best_params, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'bagging_freq': 5})
final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
proba = final_model.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, proba)
print(f'\nFinal test AUC (tuned): {final_auc:.4f}')
print('vs untuned 0.7142 (from 03)')
print(f'Improvement: {final_auc - 0.7142:+.4f}')


Running Optuna (15 trials)...
Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[92]	valid_0's binary_logloss: 0.456926
Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[192]	valid_0's binary_logloss: 0.456131


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[213]	valid_0's binary_logloss: 0.456606


...

Early stopping, best iteration is:
[32]	valid_0's binary_logloss: 0.457046
Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[385]	valid_0's binary_logloss: 0.455885

Best validation AUC: 0.7058
Best params:
  n_estimators: 1220
  learning_rate: 0.014776711980507262
  num_leaves: 208
  min_child_samples: 77
  reg_alpha: 0.00231787149016397
  reg_lambda: 0.008359260348241956
  feature_fraction: 0.6462930146050305
  bagging_fraction: 0.6470499478864152


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[272]	valid_0's binary_logloss: 0.450585



Final test AUC (tuned): 0.7176
vs untuned 0.7142 (from 03)
Improvement: +0.0034


**Observations**

- 15 Optuna trials. Best validation AUC 0.7058 (lower than the test AUC because the validation set is smaller and noisier).
- Final test AUC with best params: **0.7176**, vs **0.7142** untuned. **+0.0034 improvement**.
- Best params landed on lower learning rate (0.015 vs 0.05 default), more trees (1220 vs 500), more leaves (208 vs 63), aggressive feature/bagging fractions (~0.65). The model wants to overfit less per tree and average more — sensible for this signal-to-noise ratio.
- 15 trials is light. 50+ would tighten the result, probably another 0.001-0.002 AUC.


## Bottom line

**Tuning gain: +0.0034 AUC.**

Combined with the +0.0036 from feature engineering (notebook 04), maximum end-to-end improvement over the deployed v1 is around +0.007 AUC.

Whether to update the deployed model depends on whether 0.7 percentage points of AUC matters to the demo. For a portfolio piece where the talking point is "I tried these things and here's what they bought me," the answer is documenting the finding more than shipping the better model.
